# MobAI Warehouse Forecasting - COMPLETE DEEP DIVE

## 🎯 THE CORE OBJECTIVE

**Predict tomorrow's demand** → Generate **Preparation Orders** one day ahead → Know which products to move from STORAGE to PICKING locations.

---

## 🛠️ STEP 1: DATA EXTRACTION & JOINING

We start by loading the demand history and enriching it with product attributes. 

> [!NOTE]
> We are using the preprocessed data from `ai/data/raw/products_for_ts_grouped.csv` which already contains the joined attributes and is aggregated by day.

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Load preprocessed grouped data
df = pd.read_csv('../data/raw/products_for_ts_grouped.csv')

# Convert date to datetime
df['date'] = pd.to_datetime(df['date'])

print(f"Total records: {len(df)}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Unique products: {df['id_produit'].nunique()}")

df.head()

Total records: 80651
Date range: 2024-01-02 00:00:00 to 2026-01-08 00:00:00
Unique products: 1124


,id_produit,date,categorie,colisage fardeau,colisage palette,volume pcs (m3),Is_Gerbable,quantite_demande
0,31334,2024-01-07,MOULURE,32,1600,0.0002,True,544
1,31334,2024-01-09,MOULURE,32,1600,0.0002,True,64
2,31334,2024-01-10,MOULURE,32,1600,0.0002,True,32
3,31334,2024-01-11,MOULURE,32,1600,0.0002,True,64
4,31334,2024-01-15,MOULURE,32,1600,0.0002,True,192


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

## 🛠️ STEP 2: TEMPORAL FEATURE ENGINEERING

Demand often follows strong weekly or monthly patterns. We extract these features to help our models understand seasonality.

In [2]:
def add_temporal_features(df):
    """
    Extract temporal features that influence demand
    """
    df = df.copy()
    
    # Extract date components
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day_of_month'] = df['date'].dt.day
    df['day_of_week'] = df['date'].dt.dayofweek  # Monday=0, Sunday=6
    df['week_of_year'] = df['date'].dt.isocalendar().week
    
    # Binary flags
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    df['is_month_start'] = (df['day_of_month'] <= 7).astype(int)
    df['is_month_end'] = (df['day_of_month'] >= 24).astype(int)
    
    return df

df_with_features = add_temporal_features(df)
print("Temporal features added.")
df_with_features[['date', 'day_of_week', 'is_weekend', 'is_month_start', 'is_month_end']].head()

Temporal features added.


,date,day_of_week,is_weekend,is_month_start,is_month_end
0,2024-01-07,6,1,1,0
1,2024-01-09,1,0,0,0
2,2024-01-10,2,0,0,0
3,2024-01-11,3,0,0,0
4,2024-01-15,0,0,0,0


In [ ]:
# 📊 VISUALIZE: Temporal Patterns (Seasonality)
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Sample one product for detailed view
sample_product = df_with_features['id_produit'].value_counts().index[0]
df_sample = df_with_features[df_with_features['id_produit'] == sample_product].sort_values('date')

# 1. Demand over time with weekend highlighting
axes[0, 0].plot(df_sample['date'], df_sample['quantite_demande'], linewidth=2, color='#4ECDC4', marker='o', markersize=4)
# Highlight weekends
weekend_mask = df_sample['is_weekend'] == 1
if weekend_mask.sum() > 0:
    axes[0, 0].scatter(df_sample[weekend_mask]['date'], df_sample[weekend_mask]['quantite_demande'], 
                       color='#FF6B6B', s=100, marker='s', label='Weekend', zorder=5, alpha=0.7)
axes[0, 0].set_xlabel('Date', fontsize=10, fontweight='bold')
axes[0, 0].set_ylabel('Demand', fontsize=10, fontweight='bold')
axes[0, 0].set_title(f'Product {sample_product}: Demand Over Time', fontsize=11, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)
axes[0, 0].tick_params(axis='x', rotation=45)

# 2. Demand by day of week (all products)
weekly_pattern = df_with_features.groupby('day_of_week')['quantite_demande'].mean().sort_index()
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
colors = ['#4ECDC4' if i < 5 else '#FF6B6B' for i in range(7)]
axes[0, 1].bar(range(7), weekly_pattern.values, color=colors, alpha=0.7, edgecolor='black')
axes[0, 1].set_xticks(range(7))
axes[0, 1].set_xticklabels(day_names)
axes[0, 1].set_ylabel('Average Demand', fontsize=10, fontweight='bold')
axes[0, 1].set_title('Weekly Seasonality Pattern', fontsize=11, fontweight='bold')
axes[0, 1].grid(axis='y', alpha=0.3)

# 3. Demand by month (all products)
monthly_pattern = df_with_features.groupby('month')['quantite_demande'].mean().sort_index()
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
axes[1, 0].plot(monthly_pattern.index, monthly_pattern.values, marker='o', linewidth=2.5, markersize=8, color='#95DAC1')
axes[1, 0].set_xticks(monthly_pattern.index)
axes[1, 0].set_xticklabels([month_names[i-1] for i in monthly_pattern.index], rotation=45)
axes[1, 0].set_ylabel('Average Demand', fontsize=10, fontweight='bold')
axes[1, 0].set_title('Monthly Seasonality Pattern', fontsize=11, fontweight='bold')
axes[1, 0].grid(alpha=0.3)

# 4. Month start/end effect
labels = ['Regular Days', 'Month Start\n(1-7)', 'Month End\n(24-31)']
values = [
    df_with_features[(df_with_features['is_month_start'] == 0) & (df_with_features['is_month_end'] == 0)]['quantite_demande'].mean(),
    df_with_features[df_with_features['is_month_start'] == 1]['quantite_demande'].mean(),
    df_with_features[df_with_features['is_month_end'] == 1]['quantite_demande'].mean()
]
colors_bar = ['#4ECDC4', '#FECA57', '#FF9FF3']
axes[1, 1].bar(range(3), values, color=colors_bar, alpha=0.7, edgecolor='black')
axes[1, 1].set_xticks(range(3))
axes[1, 1].set_xticklabels(labels)
axes[1, 1].set_ylabel('Average Demand', fontsize=10, fontweight='bold')
axes[1, 1].set_title('Month-Period Effect', fontsize=11, fontweight='bold')
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🔍 SEASONALITY INSIGHTS:")
print(f"   Highest demand day: {day_names[weekly_pattern.argmax()]} ({weekly_pattern.max():.1f} avg)")
print(f"   Lowest demand day: {day_names[weekly_pattern.argmin()]} ({weekly_pattern.min():.1f} avg)")
print(f"   Weekend vs Weekday: {df_with_features[df_with_features['is_weekend']==1]['quantite_demande'].mean():.1f} vs {df_with_features[df_with_features['is_weekend']==0]['quantite_demande'].mean():.1f}")

## 🛠️ STEP 3: LAGGED & ROLLING FEATURES

Tomorrow's demand is often related to what happened yesterday or last week. We create rolling averages to capture trends.

In [3]:
def create_features(df):
    df = df.sort_values(['id_produit', 'date']).copy()
    
    # Lags
    for lag in [1, 7, 30]:
        df[f'lag_{lag}d'] = df.groupby('id_produit')['quantite_demande'].shift(lag)
    
    # Rolling means
    for window in [7, 30]:
        df[f'rolling_mean_{window}d'] = df.groupby('id_produit')['quantite_demande'].transform(
            lambda x: x.rolling(window=window, min_periods=1).mean()
        )
        
    return df.fillna(0)

df_final = create_features(df_with_features)
print("Lagged and rolling features created.")
df_final.head()

Lagged and rolling features created.


,id_produit,date,categorie,colisage fardeau,colisage palette,volume pcs (m3),Is_Gerbable,quantite_demande,year,month,...,day_of_week,week_of_year,is_weekend,is_month_start,is_month_end,lag_1d,lag_7d,lag_30d,rolling_mean_7d,rolling_mean_30d
0,31334,2024-01-07,MOULURE,32,1600,0.0002,True,544,2024,1,...,6,1,1,1,0,0.0,0.0,0.0,544.000000,544.000000
1,31334,2024-01-09,MOULURE,32,1600,0.0002,True,64,2024,1,...,1,2,0,0,0,544.0,0.0,0.0,304.000000,304.000000
2,31334,2024-01-10,MOULURE,32,1600,0.0002,True,32,2024,1,...,2,2,0,0,0,64.0,0.0,0.0,213.333333,213.333333
3,31334,2024-01-11,MOULURE,32,1600,0.0002,True,64,2024,1,...,3,2,0,0,0,32.0,0.0,0.0,176.000000,176.000000
4,31334,2024-01-15,MOULURE,32,1600,0.0002,True,192,2024,1,...,0,3,0,0,0,64.0,0.0,0.0,179.200000,179.200000


In [ ]:
# 📊 VISUALIZE: Lagged Features & Trend Capture
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Get sample product with complete data
sample_product = df_final['id_produit'].value_counts().nlargest(10).index[5]
df_sample = df_final[df_final['id_produit'] == sample_product].sort_values('date').copy()

# 1. Actual vs Lag Features
axes[0, 0].plot(df_sample.index, df_sample['quantite_demande'], label='Actual', linewidth=2.5, color='#FF6B6B', marker='o', markersize=5)
axes[0, 0].plot(df_sample.index, df_sample['lag_1d'], label='Lag 1 Day', linewidth=2, color='#4ECDC4', linestyle='--', alpha=0.7)
axes[0, 0].plot(df_sample.index, df_sample['lag_7d'], label='Lag 7 Days', linewidth=2, color='#95DAC1', linestyle=':', alpha=0.7)
axes[0, 0].set_xlabel('Record Index', fontsize=10, fontweight='bold')
axes[0, 0].set_ylabel('Demand', fontsize=10, fontweight='bold')
axes[0, 0].set_title(f'Product {sample_product}: Actual vs Lag Features', fontsize=11, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# 2. Rolling Mean - Trend Indicator
axes[0, 1].plot(df_sample.index, df_sample['quantite_demande'], label='Actual', linewidth=1.5, color='lightgray', alpha=0.6)
axes[0, 1].plot(df_sample.index, df_sample['rolling_mean_7d'], label='7-Day MA (Trend)', linewidth=2.5, color='#4ECDC4')
axes[0, 1].plot(df_sample.index, df_sample['rolling_mean_30d'], label='30-Day MA (Long Trend)', linewidth=2.5, color='#FF6B6B')
axes[0, 1].set_xlabel('Record Index', fontsize=10, fontweight='bold')
axes[0, 1].set_ylabel('Demand', fontsize=10, fontweight='bold')
axes[0, 1].set_title('Rolling Averages Capture Trend', fontsize=11, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# 3. Correlation heatmap between features
feature_cols = ['quantite_demande', 'lag_1d', 'lag_7d', 'lag_30d', 'rolling_mean_7d', 'rolling_mean_30d']
corr_matrix = df_final[feature_cols].corr()
im = axes[1, 0].imshow(corr_matrix, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
axes[1, 0].set_xticks(range(len(feature_cols)))
axes[1, 0].set_yticks(range(len(feature_cols)))
axes[1, 0].set_xticklabels([col.replace('quantite_demande', 'Actual').replace('_', ' ') for col in feature_cols], rotation=45, ha='right', fontsize=9)
axes[1, 0].set_yticklabels([col.replace('quantite_demande', 'Actual').replace('_', ' ') for col in feature_cols], fontsize=9)
axes[1, 0].set_title('Feature Correlation Matrix', fontsize=11, fontweight='bold')

# Add correlation values
for i in range(len(feature_cols)):
    for j in range(len(feature_cols)):
        text = axes[1, 0].text(j, i, f'{corr_matrix.iloc[i, j]:.2f}',
                               ha="center", va="center", color="black", fontsize=8)

fig.colorbar(im, ax=axes[1, 0])

# 4. Distribution of features
axes[1, 1].hist(df_final['quantite_demande'], bins=40, alpha=0.5, label='Actual', color='#FF6B6B', edgecolor='black')
axes[1, 1].hist(df_final['lag_1d'], bins=40, alpha=0.5, label='Lag 1D', color='#4ECDC4', edgecolor='black')
axes[1, 1].hist(df_final['rolling_mean_7d'], bins=40, alpha=0.5, label='7-Day MA', color='#95DAC1', edgecolor='black')
axes[1, 1].set_xlabel('Value', fontsize=10, fontweight='bold')
axes[1, 1].set_ylabel('Frequency', fontsize=10, fontweight='bold')
axes[1, 1].set_title('Feature Distributions Comparison', fontsize=11, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(axis='y', alpha=0.3)
axes[1, 1].set_yscale('log')

plt.tight_layout()
plt.show()

print("\n🔍 LAG & TREND INSIGHTS:")
print(f"   Correlation actual vs lag_1d: {df_final['quantite_demande'].corr(df_final['lag_1d']):.3f}")
print(f"   Correlation actual vs lag_7d: {df_final['quantite_demande'].corr(df_final['lag_7d']):.3f}")
print(f"   Correlation actual vs 7-day MA: {df_final['quantite_demande'].corr(df_final['rolling_mean_7d']):.3f}")
print(f"\n   💡 High correlation means past values predict future well!")

## 🛠️ STEP 4: PRODUCT SEGMENTATION

Not all products behave the same. Some are ordered every day in large quantities, while others are rare. We segment them to apply the best model for each.

In [4]:
def segment_products(df):
    # Calculate stats per product
    stats = df.groupby('id_produit').agg({
        'quantite_demande': ['count', 'mean', 'std'],
        'date': lambda x: (x.max() - x.min()).days
    }).reset_index()
    
    stats.columns = ['id_produit', 'order_days', 'avg_qty', 'std_qty', 'days_span']
    stats['frequency'] = stats['order_days'] / (stats['days_span'] + 1)
    
    def classify(row):
        if row['frequency'] > 0.8 and row['avg_qty'] > 500: return 'A_HIGH_FREQ_VOL'
        if row['frequency'] > 0.5: return 'B_MEDIUM_FREQ'
        return 'C_LOW_FREQ'
    
    stats['segment'] = stats.apply(classify, axis=1)
    return stats

product_segments = segment_products(df_final)
df_final = df_final.merge(product_segments[['id_produit', 'segment']], on='id_produit')
print("Product segmentation complete.")
print(df_final['segment'].value_counts())

Product segmentation complete.
segment
C_LOW_FREQ         46915
B_MEDIUM_FREQ      33702
A_HIGH_FREQ_VOL       34
Name: count, dtype: int64


In [ ]:
# 📊 VISUALIZE: Product Segmentation
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Segment distribution
segment_counts = df_final['segment'].value_counts()
colors_seg = ['#4ECDC4', '#FF6B6B', '#95DAC1']
axes[0, 0].pie(segment_counts.values, labels=segment_counts.index, autopct='%1.1f%%',
               colors=colors_seg, startangle=90, textprops={'fontsize': 10, 'fontweight': 'bold'})
axes[0, 0].set_title('Product Segments Distribution', fontsize=12, fontweight='bold')

# 2. Frequency vs Average Quantity Scatter
axes[0, 1].scatter(product_segments['frequency'], product_segments['avg_qty'], 
                   c=[colors_seg[0] if seg == 'A_HIGH_FREQ_VOL' else colors_seg[1] if seg == 'B_MEDIUM_FREQ' else colors_seg[2] 
                     for seg in product_segments['segment']], 
                   alpha=0.6, s=80, edgecolors='black', linewidth=0.5)

# Add segment boundary lines
axes[0, 1].axhline(y=500, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='Qty threshold (500)')
axes[0, 1].axvline(x=0.8, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='High freq (0.8)')
axes[0, 1].axvline(x=0.5, color='gray', linestyle=':', linewidth=1, alpha=0.5, label='Med freq (0.5)')
axes[0, 1].set_xlabel('Order Frequency', fontsize=10, fontweight='bold')
axes[0, 1].set_ylabel('Average Quantity', fontsize=10, fontweight='bold')
axes[0, 1].set_title('Product Segmentation Map', fontsize=11, fontweight='bold')
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid(alpha=0.3)

# 3. Demand characteristics by segment
segment_demand = df_final.groupby('segment').agg({
    'quantite_demande': ['mean', 'median', 'std']
}).reset_index()
segment_demand.columns = ['segment', 'mean', 'median', 'std']

x = np.arange(len(segment_demand))
width = 0.25
axes[1, 0].bar(x - width, segment_demand['mean'], width, label='Mean', color='#4ECDC4', alpha=0.7)
axes[1, 0].bar(x, segment_demand['median'], width, label='Median', color='#FF6B6B', alpha=0.7)
axes[1, 0].bar(x + width, segment_demand['std'], width, label='Std Dev', color='#95DAC1', alpha=0.7)
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(segment_demand['segment'], rotation=15, ha='right')
axes[1, 0].set_ylabel('Demand', fontsize=10, fontweight='bold')
axes[1, 0].set_title('Demand Statistics by Segment', fontsize=11, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(axis='y', alpha=0.3)

# 4. Number of products and total demand by segment
segment_stats = df_final.groupby('segment').agg({
    'id_produit': 'nunique',
    'quantite_demande': 'sum'
}).reset_index()
segment_stats.columns = ['segment', 'num_products', 'total_demand']

ax1 = axes[1, 1]
ax2 = ax1.twinx()

bars1 = ax1.bar(range(len(segment_stats)), segment_stats['num_products'], alpha=0.7, color='#4ECDC4', label='# Products')
ax2.plot(range(len(segment_stats)), segment_stats['total_demand'], marker='o', markersize=10, 
         linewidth=3, color='#FF6B6B', label='Total Demand')

ax1.set_xticks(range(len(segment_stats)))
ax1.set_xticklabels(segment_stats['segment'], rotation=15, ha='right')
ax1.set_ylabel('Number of Products', fontsize=10, fontweight='bold', color='#4ECDC4')
ax2.set_ylabel('Total Demand', fontsize=10, fontweight='bold', color='#FF6B6B')
ax1.set_title('Products vs Demand by Segment', fontsize=11, fontweight='bold')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
ax1.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🔍 SEGMENTATION INSIGHTS:")
for seg in segment_counts.index:
    seg_data = df_final[df_final['segment'] == seg]
    print(f"\n   {seg}:")
    print(f"      Products: {seg_data['id_produit'].nunique()}")
    print(f"      Avg demand: {seg_data['quantite_demande'].mean():.2f}")
    print(f"      Total demand: {seg_data['quantite_demande'].sum():,.0f}")
    print(f"      Volatility (std): {seg_data['quantite_demande'].std():.2f}")

## 🛠️ STEP 5: FORECASTING MODELS

We define our models, ranging from simple averages to Machine Learning.

In [5]:
from sklearn.ensemble import RandomForestRegressor

def get_naive_forecast(history, window=7):
    return history.tail(window).mean()

def train_rf_model(train_df, features):
    model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
    model.fit(train_df[features], train_df['quantite_demande'])
    return model

print("Model functions defined.")

Model functions defined.


## 🛠️ STEP 6: ORCHESTRATION - GENERATING THE ORDER

We combine the models to generate the final **Preparation Order** for a target date.

In [6]:
def generate_order(df, target_date):
    # Sample logic for a single product to demonstrate
    # In production, this would loop through segments and use mapped models
    history = df[df['date'] < target_date]
    
    # Naive example
    forecast = history.groupby('id_produit')['quantite_demande'].mean().reset_index()
    forecast.columns = ['id_produit', 'forecasted_quantity']
    
    return forecast

target_date = df['date'].max()
prep_order = generate_order(df_final, target_date)
print(f"Generated Preparation Order for {target_date.date()} with {len(prep_order)} items.")

Generated Preparation Order for 2026-01-08 with 1124 items.


## 📊 STEP 7: EVALUATION & VALIDATION

Finally, we evaluate our forecast against actual demand to measure accuracy (MAPE, Bias, Service Level).

In [7]:
from sklearn.metrics import mean_absolute_error

def evaluate(actual, forecast):
    mae = mean_absolute_error(actual, forecast)
    print(f"MAE: {mae:.2f} units")
    return mae

print("Evaluation logic ready.")

Evaluation logic ready.
